<a href="https://colab.research.google.com/github/roeiyanku/UAV_Sound_Classification_Project/blob/main/notebooks/04_compare_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compare Runs

Within-run analysis lives in each run's `report.html`. This notebook is for
**cross-run** comparison: did this week's experiment beat last week's?
Is augmentation actually helping? Pump vs valve?

Reads every `run_*` folder under `RESULTS_DIR` and concatenates them.


In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/Final Project RMOT/artifacts/results"

import os, json, glob
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")


## Load every run

In [ ]:
def load_all_runs(results_dir):
    rows = []
    for d in sorted(glob.glob(os.path.join(results_dir, "run_*"))):
        csv = os.path.join(d, "results.csv")
        meta_path = os.path.join(d, "metadata.json")
        if not (os.path.exists(csv) and os.path.exists(meta_path)):
            continue
        with open(meta_path) as f:
            m = json.load(f)
        sub = pd.read_csv(csv)
        sub["run_id"] = m["run_id"]
        sub["run_timestamp"] = pd.to_datetime(m["timestamp"])
        rows.append(sub)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

all_runs = load_all_runs(RESULTS_DIR)
if all_runs.empty:
    raise FileNotFoundError(f"No run folders found in {RESULTS_DIR}.")
print(f"Loaded {all_runs['run_id'].nunique()} runs, {len(all_runs)} total rows.")


## Best result per run

Did my latest run beat the previous ones?

In [ ]:
best_per_run = (
    all_runs.sort_values("auc", ascending=False)
            .groupby("run_id", as_index=False)
            .first()
            .sort_values("run_timestamp", ascending=False)
            [["run_timestamp", "run_id", "dataset", "augmented",
              "feature_name", "model_name", "auc", "f1_anomaly", "accuracy"]]
            .reset_index(drop=True)
)
best_per_run


## Best per configuration

For each `(dataset, augmented)` combination, the all-time best result across all runs.

In [ ]:
best_per_config = (
    all_runs.sort_values("auc", ascending=False)
            .groupby(["dataset", "augmented"], as_index=False)
            .first()
            [["dataset", "augmented", "run_id", "run_timestamp",
              "feature_name", "model_name", "auc", "f1_anomaly", "accuracy"]]
            .sort_values(["dataset", "augmented"])
            .reset_index(drop=True)
)
best_per_config


## Best AUC over time

One line per `(dataset, augmented)` configuration.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for (ds, aug), grp in best_per_run.groupby(["dataset", "augmented"]):
    grp = grp.sort_values("run_timestamp")
    label = f"{ds} ({'aug' if aug else 'raw'})"
    ax.plot(grp["run_timestamp"], grp["auc"], marker="o", label=label)
ax.set_xlabel("Run timestamp")
ax.set_ylabel("Best AUC in run")
ax.set_title("Best AUC per run over time")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
from google.colab import drive
drive.flush_and_unmount()